# 01 · Explore the packaged plant dataset

This notebook inspects `MyDrive/AI/meta-global-hackaton/data/plant_export_v20261026_2355546_holdout10_seed42.zip`. The package contains standard train, validation, test, and 10-per-class holdout folders, along with a manifest and class mapping. It reads images directly from the ZIP and does not modify them.

The 180-image `external_test` subset is sampled from the same source archive as training. Treat it as an internal holdout, not an independent real-world test.


## A. Dataset overview

Mount Drive, load the package metadata and manifest, and report actual counts. The packaged images use zero-based targets that match the included `class_to_idx.json`.


In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from collections import Counter
import hashlib
import itertools
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

SEED = 42
PACKAGE_NAME = 'plant_export_v20261026_2355546_holdout10_seed42.zip'
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PACKAGE_PATH = Path('/content/drive/MyDrive/AI/meta-global-hackaton/data') / PACKAGE_NAME
else:
    PACKAGE_PATH = Path('./data/processed') / PACKAGE_NAME
if not PACKAGE_PATH.is_file():
    raise FileNotFoundError(f'Packaged dataset ZIP not found: {PACKAGE_PATH}')
print('Dataset package:', PACKAGE_PATH)


## Read the package manifest

The package manifest maps each image to a split, class, ZIP path, source record, and content hash. Its source URLs are provenance metadata; they do not grant image redistribution rights.


In [ ]:
with ZipFile(PACKAGE_PATH) as archive:
    manifest = pd.read_csv(BytesIO(archive.read('manifest.csv'))).fillna('')
    metadata = json.loads(archive.read('metadata.json'))
    class_mapping = json.loads(archive.read('class_to_idx.json'))
CLASS_NAMES = class_mapping['index_to_class']
SPLIT_NAMES = ('train', 'val', 'test', 'external_test')
required_columns = {'split', 'class_name', 'class_index', 'folder_name',
                    'file_path', 'sha256', 'source_split', 'source_url'}
if not required_columns <= set(manifest.columns):
    raise ValueError(f'Manifest missing columns: {sorted(required_columns - set(manifest.columns))}')
if set(manifest.class_name) != set(CLASS_NAMES):
    raise ValueError('Manifest classes do not match class_to_idx.json')
if set(manifest.split) != set(SPLIT_NAMES):
    raise ValueError('Expected train, val, test, and external_test splits')
split_counts = manifest.split.value_counts().reindex(SPLIT_NAMES)
print('Image records:', len(manifest))
print('Classes:', len(CLASS_NAMES))
print('Split counts:', split_counts.to_dict())
print('Split fingerprint:', metadata['split_fingerprint'])
print('Exact duplicates removed during preparation:', metadata['exact_duplicate_records_removed'])
display(manifest.groupby(['split', 'class_name']).size().unstack(fill_value=0).reindex(
    index=SPLIT_NAMES, columns=CLASS_NAMES))


## B. Visual inspection

Use reproducible sampling for a 16-image grid, examples from one class, and one image from each class. Display scaling is only for visualization.


In [ ]:
def show_grid(frame: pd.DataFrame, title: str, columns: int = 4) -> None:
    """Display images from the ZIP with class and split labels."""
    if frame.empty:
        print('No images for', title)
        return
    rows = int(np.ceil(len(frame) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(columns * 3.2, rows * 3.2))
    axes = np.asarray(axes).reshape(-1)
    with ZipFile(PACKAGE_PATH) as archive:
        for axis, row in zip(axes, frame.itertuples(index=False)):
            with Image.open(BytesIO(archive.read(row.file_path))) as image:
                axis.imshow(image.convert('RGB'))
            axis.set_title(f'{row.class_name} · {row.split}', fontsize=9)
            axis.axis('off')
    for axis in axes[len(frame):]:
        axis.axis('off')
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

show_grid(manifest.sample(n=16, random_state=SEED), 'Random images · seed 42')
same_class = manifest[manifest.class_name == 'philodendron hederaceum']
show_grid(same_class.sample(n=min(6, len(same_class)), random_state=SEED),
          'Within-class variation · philodendron hederaceum', columns=3)
one_per_class = (manifest[manifest.split == 'train'].groupby('class_name', group_keys=False)
                 .sample(n=1, random_state=SEED).sort_values('class_index'))
show_grid(one_per_class, 'One training image from each class', columns=6)


## C. Class distribution

Compare class counts across the four splits and summarize the complete dataset. The 10-per-class holdout is balanced by design; the remaining splits vary with the source class sizes.


In [ ]:
counts = (manifest.groupby(['class_name', 'split']).size().unstack(fill_value=0)
          .reindex(index=CLASS_NAMES, columns=SPLIT_NAMES))
all_counts = counts.sum(axis=1)
print(pd.Series({'min': all_counts.min(), 'max': all_counts.max(),
                 'mean': all_counts.mean(), 'median': all_counts.median()},
                name='images per class').to_frame())
display(counts)
fig, ax = plt.subplots(figsize=(13, 5))
counts.plot(kind='bar', ax=ax, width=0.85)
ax.set(xlabel='Class', ylabel='Images', title='Images per class and split')
ax.tick_params(axis='x', rotation=65)
ax.legend(title='Split')
fig.tight_layout()
plt.show()


## D. Image properties

Measure image dimensions, aspect ratios, formats, color modes, and file sizes without resizing or rewriting images. The SHA-256 values are checked against the bytes in the ZIP.


In [ ]:
properties = []
with ZipFile(PACKAGE_PATH) as archive:
    for row in manifest.itertuples(index=False):
        info = archive.getinfo(row.file_path)
        data = archive.read(row.file_path)
        record = {'image_id': row.source_zip_member, 'file_path': row.file_path,
                  'split': row.split, 'class_name': row.class_name,
                  'file_size_bytes': info.file_size,
                  'sha_matches': hashlib.sha256(data).hexdigest() == row.sha256}
        try:
            with Image.open(BytesIO(data)) as image:
                image.verify()
            with Image.open(BytesIO(data)) as image:
                record.update(width=image.width, height=image.height,
                              image_format=image.format, color_mode=image.mode,
                              aspect_ratio=image.width / image.height, error='')
        except Exception as exc:
            record.update(width=np.nan, height=np.nan, image_format='', color_mode='',
                          aspect_ratio=np.nan, error=f'{type(exc).__name__}: {exc}')
        properties.append(record)
properties = pd.DataFrame(properties)
print('Hash mismatches:', int((~properties.sha_matches).sum()))
print('Unreadable images:', int(properties.error.ne('').sum()))
print('Image formats:')
display(properties.image_format.value_counts().to_frame('images'))
print('Color modes:')
display(properties.color_mode.value_counts().to_frame('images'))
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for axis, field, label in zip(axes.flat,
    ('width', 'height', 'aspect_ratio', 'file_size_bytes'),
    ('Width (pixels)', 'Height (pixels)', 'Width / height', 'File size (bytes)')):
    axis.hist(properties[field].dropna(), bins=30)
    axis.set(xlabel=label, ylabel='Images')
fig.suptitle('Image properties')
fig.tight_layout()
plt.show()


## E. Dataset quality and split overlap

Check duplicate paths, duplicate image hashes, unexpected labels, exact overlap, and near-duplicate candidates. A small perceptual-hash distance marks pairs for visual review only; it does not remove or relabel images.


In [ ]:
print('Duplicate manifest paths:', int(manifest.file_path.duplicated().sum()))
print('Unexpected class labels:', sorted(set(manifest.class_name) - set(CLASS_NAMES)))
print('Duplicate content hashes:', int(manifest.sha256.duplicated(keep=False).sum()))
print('Any hash crossing splits:', bool(manifest.groupby('sha256').split.nunique().gt(1).any()))

def difference_hash(image: Image.Image) -> int:
    """Return a 64-bit difference hash for candidate screening."""
    pixels = np.asarray(image.convert('L').resize((9, 8)))
    bits = pixels[:, 1:] > pixels[:, :-1]
    result = 0
    for bit in bits.flat:
        result = (result << 1) | int(bit)
    return result

hashes = []
with ZipFile(PACKAGE_PATH) as archive:
    for row in manifest.itertuples(index=False):
        with Image.open(BytesIO(archive.read(row.file_path))) as image:
            hashes.append(difference_hash(image))
near = []
for left, right in itertools.combinations(range(len(manifest)), 2):
    distance = (hashes[left] ^ hashes[right]).bit_count()
    if distance <= 4:
        a, b = manifest.iloc[left], manifest.iloc[right]
        near.append({'left_class': a.class_name, 'left_split': a.split,
                     'left_path': a.file_path, 'right_class': b.class_name,
                     'right_split': b.split, 'right_path': b.file_path,
                     'hash_distance': distance})
near = pd.DataFrame(near)
print('Near-duplicate candidates (distance <= 4):', len(near))
if not near.empty:
    display(near.sort_values(['left_split', 'hash_distance']).head(30))


## F. Split distribution

The original source folder membership is kept only as provenance. Compare it with the packaged train, validation, test, and internal holdout assignments. The split is stratified by class, and every class contributes 10 images to the internal holdout.


In [ ]:
source_distribution = (manifest.groupby(['class_name', 'source_split']).size()
                       .unstack(fill_value=0).reindex(CLASS_NAMES))
new_distribution = counts.reindex(CLASS_NAMES)
print('Original source folders represented in the new package:')
display(pd.crosstab(manifest.split, manifest.source_split))
display(new_distribution)
assert manifest[manifest.split == 'external_test'].groupby('class_name').size().eq(10).all()


## G. Findings from this run

- **Observations:** The package contains 1,292 images across 18 classes: 890 train, 111 validation, 111 test, and 180 internal holdout images. The holdout has 10 images per class. Class totals range from 42 to 94 (median 73). All 1,292 files opened as RGB JPEGs, and every SHA-256 matched the manifest.
- **Dataset limitations:** This is a closed set of 18 classes from one source collection. Class counts are uneven, and the 180-image holdout comes from the same archive as training, so it is not an independent real-world test. Source URLs provide provenance; image permissions and botanical labels still require review.
- **Potential leakage:** No exact duplicate hashes or exact overlap across splits were found. The perceptual-hash check found 10 candidate pairs at distance 4 or less; four cross split boundaries, and two of those have distance 0. Inspect those pairs before interpreting test or holdout scores. These are candidates, not confirmed duplicate photographs.
- **Next experiments:** Visually review the cross-split candidates and uncertain labels, train the three PyTorch architectures using train and validation only, evaluate the 111-image test split once, then report the 180-image holdout separately as an internal holdout.
